In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np

from vetuner.gating import GatingConfig, implausible_afr_mask, out_of_domain_mask, overrun_mask
from vetuner.surface_fit import cell_sample_counts

config = GatingConfig()
valid = ~(implausible_afr_mask(log, config) | overrun_mask(log, config) | out_of_domain_mask(log))

counts = cell_sample_counts(log.rpm[valid], log.map_kpa[valid])
print(f"coverage ceiling with no transient gating: {(counts > 0).mean():.0%} "
      f"({(counts > 0).sum()} of {counts.size} cells)")
print(f"cells with 20+ samples: {(counts >= 20).sum()}")

In [ ]:
from dataclasses import replace

from vetuner.correction import gated_correction
from vetuner.metrics import evaluate_afr, evaluate_table

print(f"{'displ':>6} {'settle':>7} {'accept':>7} {'cover':>6} {'RMSE':>7} "
      f"{'visited':>8} {'max':>7} {'AFR%':>6}")

for displacement in (0.1, 0.2, 0.3, 0.5, 0.75, 1.0):
    for settle in (0.2, 0.4):
        cfg = replace(config, max_cell_displacement=displacement, settle_time_s=settle)
        corrected, gate_result = gated_correction(log, base, gating=cfg)

        t = evaluate_table(corrected.table, truth, corrected.counts)
        a = evaluate_afr(corrected.table, test)
        print(f"{displacement:6.2f} {settle:7.2f} {gate_result.acceptance_rate:7.0%} "
              f"{t.coverage:6.0%} {t.rmse:7.2f} {t.rmse_visited:8.2f} "
              f"{t.max_abs_error:7.2f} {a.mean_abs_pct:6.2f}")

In [ ]:
from vetuner.correction import baseline_correction, gated_correction
from vetuner.drive_cycle import VALIDATION_CYCLE, generate_drive_cycle
from vetuner.engine_model import simulate_engine
from vetuner.metrics import evaluate_afr, evaluate_table
from vetuner.sensors import apply_sensor_model
from vetuner.ve_surface import BASE_MAP_SURFACE, TRUE_SURFACE, ve_table

train = generate_drive_cycle()
test = generate_drive_cycle(cycle=VALIDATION_CYCLE)
base, truth = ve_table(params=BASE_MAP_SURFACE), ve_table(params=TRUE_SURFACE)

sim = simulate_engine(train, base)
log = apply_sensor_model(sim, seed=0)

naive = baseline_correction(log, base)
gated, gate = gated_correction(log, base)

print(f"accepted {gate.n_accepted} of {gate.n_total} ({gate.acceptance_rate:.0%})")
for name, count in gate.rejection_summary().items():
    print(f"  {name:18s} {count:6d}")

print()
for name, table, counts in [
    ("starting map", base, gated.counts),
    ("naive baseline", naive.table, naive.counts),
    ("gated", gated.table, gated.counts),
]:
    t = evaluate_table(table, truth, counts)
    a = evaluate_afr(table, test)
    print(f"{name:16s} RMSE {t.rmse_visited:5.2f} pts   max {t.max_abs_error:5.2f}   "
          f"held-out AFR {a.mean_abs_pct:5.2f}%   coverage {t.coverage:.0%}")